In [2]:
!rm -rf /content/drive


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)



Mounted at /content/drive


In [4]:
BASE = "/content/drive/MyDrive/Just_Advisor_Ai"
print(BASE)


/content/drive/MyDrive/Just_Advisor_Ai


In [5]:
import os

for f in ["data", "transcripts", "structured_arguments"]:
    os.makedirs(f"{BASE}/{f}", exist_ok=True)

print("Folders ready")


Folders ready


In [6]:
import json
from datetime import datetime

transcript = []

def add(speaker, text):
    transcript.append({
        "speaker": speaker,
        "time": datetime.now().strftime("%H:%M:%S"),
        "text": text
    })

# Example
add("Lawyer_A", "The accused violated Section 420 IPC.")
add("Lawyer_B", "There is no fraudulent intention.")
add("Lawyer_A", "Bank records prove deception.")

path = f"{BASE}/transcripts/case1.json"

with open(path, "w") as f:
    json.dump(transcript, f, indent=2)

print("Saved to:", path)


Saved to: /content/drive/MyDrive/Just_Advisor_Ai/transcripts/case1.json


In [7]:
!pip install transformers torch nltk


In [9]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [10]:
import json, nltk
nltk.download("punkt")
from nltk.tokenize import sent_tokenize

with open(f"{BASE}/transcripts/case1.json") as f:
    transcript = json.load(f)

sentences = []

for t in transcript:
    for s in sent_tokenize(t["text"]):
        sentences.append({"speaker": t["speaker"], "sentence": s})

sentences


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[{'speaker': 'Lawyer_A', 'sentence': 'The accused violated Section 420 IPC.'},
 {'speaker': 'Lawyer_B', 'sentence': 'There is no fraudulent intention.'},
 {'speaker': 'Lawyer_A', 'sentence': 'Bank records prove deception.'}]

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=5)

labels = ["Claim", "Evidence", "LegalRule", "Rebuttal", "Conclusion"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def classify(sentence):
    inp = tokenizer(sentence, return_tensors="pt", truncation=True)
    with torch.no_grad():
        out = model(**inp)
    return labels[torch.argmax(out.logits).item()]

structured = []

for s in sentences:
    structured.append({
        "speaker": s["speaker"],
        "sentence": s["sentence"],
        "role": classify(s["sentence"])
    })

structured


[{'speaker': 'Lawyer_A',
  'sentence': 'The accused violated Section 420 IPC.',
  'role': 'Rebuttal'},
 {'speaker': 'Lawyer_B',
  'sentence': 'There is no fraudulent intention.',
  'role': 'Rebuttal'},
 {'speaker': 'Lawyer_A',
  'sentence': 'Bank records prove deception.',
  'role': 'Claim'}]

In [13]:
with open(f"{BASE}/structured_arguments/case1.json", "w") as f:
    json.dump(structured, f, indent=2)

print("Structured arguments saved.")


Structured arguments saved.


In [15]:
!ls /content/drive/MyDrive/Just_Advisor_Ai/data


argument_dataset_indian_law.jsonl
